<!--
Copyright (c) 2026 OceanBase.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 22 · 一次真实任务，从接手到团队复用

订单 CSV 导入器的金额函数存在边界缺陷。Agent A 先复现问题，留下有证据的交接；Agent B 在新进程中接手，修改代码并完成检查；经过审核的经验生成 Skill，Receiver 把它交付到另一个 Agent 目录，Agent C 在新会话中使用它。

这是一篇可以独立运行的综合示例。需要真实 Generation、支持工具调用的对话模型，以及 notebooks 依赖。预计 10–20 分钟，取决于模型。代码、测试、Agent 消息、Source、准确版本、安装文件和报告都可复查。

我们只在本次新建项目中执行模型生成的 amount.py，不修改仓库业务文件。三位 Agent 是三个真实 Python/LLM 工具进程；不冒充 Codex 或 Claude Code 原生客户端。Receiver 安装到标准 Agent Skill 目录，工具进程会实际读取安装后的 SKILL.md。

路线：项目约定 → 复现 → 交接 → 修复与验收 → Task Outcome → Experience → Skill → 分发与使用 → 团队共享和报告。

In [ ]:
import sys
from pathlib import Path

from _tutorial import Tutorial, show, table

from powercontext.http import CreateScopeRequest

if not Path("_tutorial.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "examples" / "jupyter"))
if previous_lab := globals().get("lab"):
    await previous_lab.close()
lab = await Tutorial.start("22", features=("generation",))
client = lab.client
assert client is not None
scope = await client.create_scope(
    CreateScopeRequest(
        title="订单 CSV 导入器 · 22", summary="本次教学实验的独立材料", idempotency_key=f"{lab.run_id}:main"
    )
)
scope_id = scope.scope_id

## 项目事实与可运行的失败

约定通过 Memory 提供给后续请求；实现和测试通过真实文件提供。测试由本篇固定，Agent 只能修改 amount.py，不能把验收标准一起改掉。

In [ ]:
import asyncio
import hashlib
import json

from powercontext.http import CreateSourceRequest, RememberMemoryRequest, SourceReference

support = Path(sys.modules["_tutorial"].__file__).parent / "support"
project = lab.directory / "csv-project"
project.mkdir()
(project / "amount.py").write_text(
    "from decimal import Decimal\ndef cents(text):\n    return int(Decimal(text) * 100)\n", encoding="utf-8"
)
(project / "test_amount.py").write_bytes((support / "test_amount_fixture.txt").read_bytes())
tests_digest = hashlib.sha256((project / "test_amount.py").read_bytes()).hexdigest()
await client.remember_memory(
    RememberMemoryRequest(
        scope_id=scope_id,
        kind="constraint",
        text="amount: cents(text) 将非负有限金额转换为整数分；超精度、负数、非有限值和非法文本必须抛 ValueError。保持测试不变。",
        reason="本篇用户明确确认的任务边界",
    )
)


async def invoke_agent(task, can_edit=False, skill_path=None):
    process = await asyncio.create_subprocess_exec(
        sys.executable,
        str(support / "amount_agent.py"),
        stdin=asyncio.subprocess.PIPE,
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE,
    )
    payload = {
        "workspace": str(project),
        "task": task,
        "can_edit": can_edit,
        "scope_id": scope_id,
        "base_url": lab.base_url,
        "skill_path": str(skill_path) if skill_path else None,
    }
    try:
        stdout, _stderr = await asyncio.wait_for(process.communicate(json.dumps(payload).encode()), 240)
    except TimeoutError:
        process.kill()
        await process.wait()
        raise
    assert process.returncode == 0, "Agent 执行失败，请检查模型配置与工具调用能力"
    result = json.loads(stdout)
    (lab.directory / f"agent-{result['pid']}.json").write_text(json.dumps(result, ensure_ascii=False, indent=2))
    return result


agent_a = await invoke_agent(
    "amount: 读取项目约定，inspect_project 并 run_checks。只复现和诊断金额转换问题，留下明确下一步，不修改文件。"
)
checks_a = [m for m in agent_a["messages"] if m["type"] == "tool" and m.get("name") == "run_checks"]
assert checks_a and json.loads(checks_a[-1]["content"])["exit_code"] != 0
print(agent_a["answer"])

## 把 Agent A 的现场变成可检查的交接

Source 保存实际工具结果与文件摘要；Handoff 只陈述这些证据支持的状态。预览不会提交制品，commit 才产生接收方可以指定的 Revision。

In [ ]:
from powercontext.http import CommitHandoffRequest, HandoffCurrentWorkRequest, ListArtifactsRequest

code_digest = hashlib.sha256((project / "amount.py").read_bytes()).hexdigest()
evidence_a = await client.create_source(
    scope_id,
    CreateSourceRequest(
        content={
            "agent_pid": agent_a["pid"],
            "checks": checks_a,
            "amount_sha256": code_digest,
            "test_sha256": tests_digest,
        }
    ),
)
citation_a = {"kind": "source", "source_ref": {"name": "content", "source_id": evidence_a.source_id}}
objective = "修复 amount 金额转换并通过本篇正常值与非法值测试"
preview = await client.handoff_current_work(
    HandoffCurrentWorkRequest.model_validate({
        "scope_id": scope_id,
        "source_id": f"{lab.run_id}:handoff",
        "handoff": {
            "schema": "powercontext.current-work-handoff.v1",
            "trust": "untrusted_input",
            "objective": objective,
            "state": [
                {
                    "text": "已运行测试，当前实现未通过非法金额检查；尚未修改代码。",
                    "basis": "verified",
                    "evidence": [citation_a],
                }
            ],
            "disposition": "continuable",
            "next_action": {
                "text": "读取文件与测试，修复 cents(text)，重新运行测试并记录结果。",
                "basis": "declared",
                "evidence": [],
            },
            "omissions": ["未验证真实业务 CSV 文件或吞吐性能。"],
        },
    })
)
assert not (await client.list_artifacts(scope_id, "handoff", ListArtifactsRequest())).items
show(preview.handoff)
committed = await client.commit_handoff(CommitHandoffRequest(scope_id=scope_id, handoff=preview.handoff))
handoff_ref = committed.reference

## Agent B 从准确版本恢复，在新进程中继续

先精确读取交接，复查文件摘要，再确认接收。Agent B 得到的历史由交接内容提供，不包含 Agent A 的整个会话；它还会重新读取实际项目和运行测试。

In [ ]:
from powercontext.http import AcknowledgeHandoffRequest, ContinueHandoffRequest

incoming = await client.continue_handoff(
    ContinueHandoffRequest(scope_id=scope_id, selection="exact", revision=handoff_ref)
)
assert incoming.content and incoming.selected_revision == handoff_ref
assert hashlib.sha256((project / "amount.py").read_bytes()).hexdigest() == code_digest
receipt = await client.acknowledge_handoff(
    AcknowledgeHandoffRequest.model_validate({
        "scope_id": scope_id,
        "source_id": f"{lab.run_id}:accepted",
        "receiver": "tutorial-agent-b",
        "status": "accepted",
        "selection": "exact",
        "revision": handoff_ref.model_dump(),
        "receiver_checks": {"live_state": "confirmed", "capability": "confirmed", "authorization": "confirmed"},
        "message": "已确认教学目录摘要；本次授权只修改 amount.py 并运行原测试。",
    })
)
agent_b = await invoke_agent(
    "amount: 请接续这份历史并复查现场："
    + incoming.content.model_dump_json()
    + "。修复 cents(text)，只修改 amount.py。必须运行测试，直到本篇验收通过。",
    can_edit=True,
)
assert agent_b["pid"] != agent_a["pid"]
assert any(call["name"] == "write_amount_module" for m in agent_b["messages"] for call in m.get("tool_calls", []))
assert hashlib.sha256((project / "test_amount.py").read_bytes()).hexdigest() == tests_digest
process = await asyncio.create_subprocess_exec(
    sys.executable,
    "-m",
    "unittest",
    "-v",
    "test_amount",
    cwd=project,
    stdout=asyncio.subprocess.PIPE,
    stderr=asyncio.subprocess.PIPE,
)
stdout, stderr = await asyncio.wait_for(process.communicate(), 30)
assert process.returncode == 0, (stdout + stderr).decode()
print((stdout + stderr).decode())
evidence_b = await client.create_source(
    scope_id,
    CreateSourceRequest(
        content={
            "agent_pid": agent_b["pid"],
            "actual_exit_code": process.returncode,
            "checks": (stdout + stderr).decode(),
            "implementation": (project / "amount.py").read_text(),
            "test_sha256": tests_digest,
        }
    ),
)
citation_b = {"kind": "source", "source_ref": {"name": "content", "source_id": evidence_b.source_id}}

## 记录这次任务的结果

通过的是本篇金额函数验收，不是所有 CSV 功能。Task Outcome 保存这个范围和真实证据，再供经验生成使用。

In [ ]:
from powercontext.http import RecordTaskOutcomeRequest

outcome = await client.record_task_outcome(
    RecordTaskOutcomeRequest.model_validate({
        "scope_id": scope_id,
        "source_id": f"{lab.run_id}:completed",
        "outcome": {
            "schema": "powercontext.task-outcome.v1",
            "trust": "untrusted_observation",
            "objective": objective,
            "status": "succeeded",
            "summary": "金额函数的本篇测试已通过；未声称真实业务吞吐或完整 CSV 接入通过。",
            "handoff_receipt_ref": receipt.receipt.source.model_dump(),
            "observations": [
                {
                    "text": "独立 unittest 进程返回 0，测试文件摘要未改变。",
                    "basis": "verified",
                    "evidence": [citation_b],
                }
            ],
            "checks": [
                {"name": "amount 正常值与非法值验收", "status": "passed", "basis": "verified", "evidence": [citation_b]}
            ],
            "produced_artifacts": [],
            "remaining_work": [],
        },
    })
)
show({"任务结果 Source": outcome.source.model_dump(), "实际退出码": process.returncode})

## 先审经验，再生成 Skill

生成模型提出的结论仍是候选。逐次显示建议，检查引用和适用范围，然后在本教学实验中明确批准。Skill 采用 experience 来源，保留准确 Experience 版本。

In [ ]:
from powercontext.http import ApproveArtifactCandidateRequest, GenerateExperienceRequest, GenerateSkillRequest

experience = await client.generate_experience(
    GenerateExperienceRequest(
        scope_id=scope_id,
        source_refs=[outcome.source, SourceReference(name="content", source_id=evidence_b.source_id)],
        artifact_refs=[],
        reason="Summarize the verified amount checks, preserve limitations, and do not invent throughput or CSV integration results.",
    )
)
assert experience.candidate and experience.candidate.status == "pending"
show(experience.candidate.proposal)
approved_experience = await client.approve_artifact_candidate(
    ApproveArtifactCandidateRequest(
        scope_id=scope_id, candidate_id=experience.candidate.candidate_id, expected_version=experience.candidate.version
    )
)
experience_ref = approved_experience.result_artifact
assert experience_ref
skill = await client.generate_skill(
    GenerateSkillRequest(
        scope_id=scope_id,
        origin="experience",
        source_refs=[],
        artifact_refs=[experience_ref],
        reason="Generate a Skill named csv-team-amount-check for inspecting project constraints, preserving tests, running actual amount boundary checks, and reporting limited evidence. Do not claim all CSV workflows are validated.",
    )
)
assert skill.candidate and skill.candidate.status == "pending"
show(skill.candidate.proposal)
approved_skill = await client.approve_artifact_candidate(
    ApproveArtifactCandidateRequest(
        scope_id=scope_id, candidate_id=skill.candidate.candidate_id, expected_version=skill.candidate.version
    )
)
skill_ref = approved_skill.result_artifact
assert skill_ref

## 交付到实际 Agent 目录

新建 Receiver target 并发布准确 Skill 版本。Receiver 在独立进程中下载并安装；输出中只保留非敏感回执，不显示 enrollment 或凭证。

In [ ]:
from powercontext.http import (
    CreateRemoteSkillTargetRequest,
    EnrollRemoteSkillTargetRequest,
    GetSkillPackageRequest,
    PublishRemoteSkillRequest,
)

receiver_project = lab.directory / "agent-c"
receiver_project.mkdir()
enrollment = await client.create_remote_skill_target(
    CreateRemoteSkillTargetRequest(scope_id=scope_id, agent_kind="codex", display_name="综合篇 Agent C")
)
credential = await client.enroll_remote_skill_target(
    EnrollRemoteSkillTargetRequest(
        enrollment_code=enrollment.enrollment_code, installation_id=lab.run_id, receiver_version="0.1.0"
    )
)
publication = await client.publish_remote_skill(
    PublishRemoteSkillRequest(
        scope_id=scope_id, target_id=credential.target_id, artifact=skill_ref, expected_generation=None
    )
)
process = await asyncio.create_subprocess_exec(
    sys.executable,
    str(support / "receiver_worker.py"),
    stdin=asyncio.subprocess.PIPE,
    stdout=asyncio.subprocess.PIPE,
    stderr=asyncio.subprocess.PIPE,
)
receiver_config = {
    "server_url": lab.base_url,
    "target_id": credential.target_id,
    "credential": credential.credential,
    "agent_kind": "codex",
    "workspace": str(receiver_project),
}
stdout, stderr = await asyncio.wait_for(process.communicate(json.dumps(receiver_config).encode()), 90)
assert process.returncode == 0
sync_result = json.loads(stdout)
assert sync_result["succeeded"] == 1 and sync_result["failed"] == 0
manifest = await client.get_skill_package_manifest(GetSkillPackageRequest(scope_id=scope_id, artifact=skill_ref))
installed = receiver_project / ".agents" / "skills" / manifest.name / "SKILL.md"
assert installed.is_file()
entry = next(item for item in manifest.files if item.path == "SKILL.md")
assert hashlib.sha256(installed.read_bytes()).hexdigest() == entry.digest
show({"安装文件": str(installed), "实际 Receiver 结果": sync_result})

## Agent C 在新任务中使用安装后的 Skill

第三个进程没有前两次消息历史。它必须实际读取选中的安装文件并执行检查。使用记录引用这个精确版本、包摘要与实际执行证据。

In [ ]:
from powercontext.http import RecordSkillUsageRequest

agent_c = await invoke_agent(
    "amount: 按已安装 Skill 完成一次独立复查。先 inspect_project，再 run_checks。只检查，不修改代码，报告验证范围。",
    skill_path=installed,
)
assert len({agent_a["pid"], agent_b["pid"], agent_c["pid"]}) == 3
inspection = [m for m in agent_c["messages"] if m["type"] == "tool" and m.get("name") == "inspect_project"]
assert inspection and "selected_skill" in json.loads(inspection[0]["content"])
checks_c = [m for m in agent_c["messages"] if m["type"] == "tool" and m.get("name") == "run_checks"]
assert checks_c and json.loads(checks_c[-1]["content"])["exit_code"] == 0
used = await client.create_source(
    scope_id,
    CreateSourceRequest(
        content={"agent_pid": agent_c["pid"], "actual_checks": checks_c, "skill_entry_sha256": entry.digest}
    ),
)
await client.record_skill_usage(
    RecordSkillUsageRequest(
        scope_id=scope_id,
        observation_id=f"{lab.run_id}:agent-c",
        skill_ref=skill_ref,
        package_digest="sha256:" + manifest.package.tree_digest,
        target_id=credential.target_id,
        selected=True,
        invoked="true",
        validation="passed",
        outcome="success",
        task_source=SourceReference(name="content", source_id=used.source_id),
    )
)
print(agent_c["answer"])

## 明确提交完成里程碑

Task Outcome 保存执行结果，不会自动修改先前的交接。现在将独立验收证据写入新的 complete Handoff，团队报告才能显示当前已完成状态，同时保留 Agent A 的原始历史版本。

In [ ]:
finished = await client.handoff_current_work(
    HandoffCurrentWorkRequest.model_validate({
        "scope_id": scope_id,
        "source_id": f"{lab.run_id}:finished-handoff",
        "handoff": {
            "schema": "powercontext.current-work-handoff.v1",
            "trust": "untrusted_input",
            "objective": objective,
            "state": [{"text": "原测试保持不变，独立验收通过。", "basis": "verified", "evidence": [citation_b]}],
            "disposition": "complete",
            "next_action": None,
            "omissions": ["没有验证业务吞吐、其他货币或完整 CSV 工作流。"],
        },
    })
)
final_commit = await client.commit_handoff(CommitHandoffRequest(scope_id=scope_id, handoff=finished.handoff))
assert final_commit.reference.artifact_id == handoff_ref.artifact_id
assert final_commit.reference.revision > handoff_ref.revision
historical = await client.get_artifact_revision(scope_id, "handoff", handoff_ref.artifact_id, handoff_ref.revision)
assert historical.content["disposition"] == "continuable"
show({"历史交接": handoff_ref.revision, "完成里程碑": final_commit.reference.revision})

## 分享经验，并查看团队报告

将审核后的准确 Experience 发布到另一项目，检查副本与原版本一致。报告展示已提交的交接里程碑；Task Outcome 与交接是独立记录，结果写入不会悄悄改写之前的 Handoff。

In [ ]:
from powercontext.http import ArtifactAddress, GetHandoffReportRequest, PublishArtifactRequest, ScopeSelection

team = await client.create_scope(
    CreateScopeRequest(title="团队复用项目", summary="接收本次验收的准确经验版本", idempotency_key=f"{lab.run_id}:team")
)
shared = await client.publish_artifact(
    PublishArtifactRequest(
        source=ArtifactAddress(scope_id=scope_id, artifact=experience_ref),
        target_scope_id=team.scope_id,
        idempotency_key=f"{lab.run_id}:share",
    )
)
original = await client.get_artifact_revision(
    scope_id, "experience", experience_ref.artifact_id, experience_ref.revision
)
copy = await client.get_artifact_revision(
    team.scope_id, "experience", shared.target.artifact.artifact_id, shared.target.artifact.revision
)
assert copy.content == original.content
report = await client.get_handoff_report(
    GetHandoffReportRequest(
        selection=ScopeSelection.model_validate({"mode": "exact", "scope_ids": [scope_id, team.scope_id]}),
        format="markdown",
    )
)
assert isinstance(report, str) and report
(lab.directory / "team-report.md").write_text(report)
print(report)
table([
    {"阶段": "Agent A", "证据": "独立进程复现失败"},
    {"阶段": "Agent B", "证据": "精确交接、修改代码、原测试通过"},
    {"阶段": "知识沉淀", "证据": "Task Outcome、已审核 Experience 与 Skill"},
    {"阶段": "Agent C", "证据": "读取实际安装文件并重跑检查"},
    {"阶段": "团队复用", "证据": "准确 Experience 副本与报告"},
])

## 收尾：撤回本次交付并撤销 Receiver

只删除本次 target 安装的、摘要仍匹配的包。实验数据库和项目证据保留供检查；如果使用临时 OceanBase 验收脚本，则由创建数据库的脚本负责删除它。

In [ ]:
from powercontext.http import ListRemoteSkillTargetsRequest, RevokeRemoteSkillTargetRequest, UnpublishRemoteSkillRequest

await client.unpublish_remote_skill(
    UnpublishRemoteSkillRequest(
        scope_id=scope_id,
        target_id=credential.target_id,
        artifact_id=skill_ref.artifact_id,
        expected_generation=publication.generation,
    )
)
process = await asyncio.create_subprocess_exec(
    sys.executable,
    str(support / "receiver_worker.py"),
    stdin=asyncio.subprocess.PIPE,
    stdout=asyncio.subprocess.PIPE,
    stderr=asyncio.subprocess.PIPE,
)
stdout, stderr = await asyncio.wait_for(process.communicate(json.dumps(receiver_config).encode()), 90)
assert process.returncode == 0 and not installed.exists()
targets = await client.list_remote_skill_targets(
    ListRemoteSkillTargetsRequest(scope_id=scope_id, target_id=credential.target_id)
)
await client.revoke_remote_skill_target(
    RevokeRemoteSkillTargetRequest(
        scope_id=scope_id, target_id=credential.target_id, expected_generation=targets.targets[0].target.generation
    )
)

## 练习与验收

给金额函数增加一条新需求，再从实际失败开始走一遍：先有执行证据，再更新经验或 Skill。判断是否完成时，分别检查任务结果、候选审核、安装回执和实际使用；不要用其中一个状态代替全部。

最后关闭服务。实验文件保留在本次 `.powercontext/` 目录，便于复查。

In [ ]:
await lab.close()
print("本篇 Server 已关闭。")